# 03 - NOAA Storm Data Pipeline

Loads NOAA Storm Events data (2020-2025), cleans and aggregates to county-month level.

**Input:** NOAA Storm Events CSV files (details + locations) fetched directly from NCEI

**Output:** `../data/processed/storm_events.pkl` — one row per county-month with at least one storm event

**Severity measures:**
- `event_count` — number of NOAA event records in that county-month
- `total_damage` — summed property + crop damage in dollars
- `log_damage` — log(total_damage + 1) to handle zeros and compress scale
- `total_duration_days` — summed storm duration across all events in county-month

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import urllib.request

Path('../data/raw').mkdir(parents=True, exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)

## NOAAStormParser
Unchanged from `NOAA_Storm_and_Zillow_Data_Pipeline.ipynb`

In [2]:
class NOAAStormParser:
    def __init__(self, details_url, locations_url):
        details_df = pd.read_csv(details_url)
        locations_df = pd.read_csv(locations_url)
        self.df = pd.merge(locations_df, details_df, on="EVENT_ID", how="left")
        self.clean()
        
    def clean(self):
        self.parse_dates()
        self.parse_damages()
        self.parse_storm_duration()

    def filter_events(self, event_types):
        return self.df[self.df['EVENT_TYPE'].isin(event_types)]

    def parse_dates(self):
        self.df["YEARMONTH"] = pd.to_datetime(self.df["YEARMONTH"], format='%Y%m') + pd.offsets.MonthEnd(0)
        
    def parse_storm_duration(self):
        self.df["BEGIN_DATE_TIME"] = pd.to_datetime(self.df["BEGIN_DATE_TIME"], format='mixed')
        self.df["END_DATE_TIME"] = pd.to_datetime(self.df["END_DATE_TIME"], format='mixed')
        duration_delta = self.df["END_DATE_TIME"] - self.df["BEGIN_DATE_TIME"]
        self.df["DURATION_HOURS"] = duration_delta.dt.total_seconds() / 3600.0
        
    def parse_damages(self):
        self.df['DAMAGE_PROPERTY'] = self.df["DAMAGE_PROPERTY"].apply(self.parse_numeric_string)
        self.df['DAMAGE_CROPS'] = self.df["DAMAGE_CROPS"].apply(self.parse_numeric_string)
        self.df['TOTAL_DAMAGE'] = self.df["DAMAGE_CROPS"] + self.df["DAMAGE_PROPERTY"]
        
    def parse_numeric_string(self, rawstring):
        if pd.isna(rawstring):
            return 0
        rawstring = str(rawstring) 
        if "K" in rawstring:
            return float(rawstring.replace("K", "")) * 1_000
        if "M" in rawstring:
            return float(rawstring.replace("M", "")) * 1_000_000
        if "B" in rawstring:
            return float(rawstring.replace("B", "")) * 1_000_000_000
        try:
            return float(rawstring)
        except ValueError:
            return 0

## Load all years

In [3]:
NOAA_URLS = {
    2020: {
        'details':   'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2020_c20260323.csv.gz',
        'locations': 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2020_c20260323.csv.gz',
    },
    2021: {
        'details':   'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2021_c20260323.csv.gz',
        'locations': 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2021_c20260323.csv.gz',
    },
    2022: {
        'details':   'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2022_c20260323.csv.gz',
        'locations': 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2022_c20260323.csv.gz',
    },
    2023: {
        'details':   'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2023_c20260323.csv.gz',
        'locations': 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2023_c20260323.csv.gz',
    },
    2024: {
        'details':   'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2024_c20260323.csv.gz',
        'locations': 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2024_c20260323.csv.gz',
    },
    2025: {
        'details':   'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2025_c20260323.csv.gz',
        'locations': 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2025_c20260323.csv.gz',
    },
}

SEVERE_EVENTS =  ['Flash Flood', 'Hail', 'Tornado', 'Flood']
SEVERE_EVENTS =  [ 'Hail']

def download_if_missing(url, local_path):
    if not local_path.exists():
        print(f'  Downloading {local_path.name}...')
        urllib.request.urlretrieve(url, local_path)
    else:
        print(f'  Using cached {local_path.name}')

frames = []
for year, urls in NOAA_URLS.items():
    print(f'Loading {year}...')
    details_path   = Path(f'../data/raw/StormEvents_details_{year}.csv.gz')
    locations_path = Path(f'../data/raw/StormEvents_locations_{year}.csv.gz')
    
    download_if_missing(urls['details'], details_path)
    download_if_missing(urls['locations'], locations_path)
    
    parser = NOAAStormParser(str(details_path), str(locations_path))
    parser.df['DATA_YEAR'] = year
    frames.append(parser.filter_events(SEVERE_EVENTS))
    print(f'  {len(parser.df):,} records')

raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal raw records: {len(raw):,}')

Loading 2020...
  Using cached StormEvents_details_2020.csv.gz
  Using cached StormEvents_locations_2020.csv.gz
  62,421 records
Loading 2021...
  Using cached StormEvents_details_2021.csv.gz
  Using cached StormEvents_locations_2021.csv.gz
  52,192 records
Loading 2022...
  Using cached StormEvents_details_2022.csv.gz
  Using cached StormEvents_locations_2022.csv.gz
  47,530 records
Loading 2023...
  Using cached StormEvents_details_2023.csv.gz
  Using cached StormEvents_locations_2023.csv.gz
  43,572 records
Loading 2024...
  Using cached StormEvents_details_2024.csv.gz
  Using cached StormEvents_locations_2024.csv.gz
  41,603 records
Loading 2025...
  Using cached StormEvents_details_2025.csv.gz
  Using cached StormEvents_locations_2025.csv.gz
  51,870 records

Total raw records: 37,422


## Build FIPS and filter to county-level events

In [4]:
# NOAA uses STATE_FIPS (2-digit) and CZ_FIPS (3-digit) for county zone events
# CZ_TYPE == 'C' means county; 'Z' means forecast zone (not county-level)
county_events = raw[raw['CZ_TYPE'] == 'C'].copy()

# Build zero-padded 5-digit stcofips
county_events['state_fips'] = county_events['STATE_FIPS'].astype(int).astype(str).str.zfill(2)
county_events['county_fips'] = county_events['CZ_FIPS'].astype(int).astype(str).str.zfill(3)
county_events['stcofips'] = county_events['state_fips'] + county_events['county_fips']

# Extract year and month from YEARMONTH
county_events['year'] = county_events['YEARMONTH'].dt.year
county_events['month'] = county_events['YEARMONTH'].dt.month

print(f'County-level records: {len(county_events):,}')
print(f'Unique counties: {county_events["stcofips"].nunique():,}')
print(f'Year range: {county_events["year"].min()} - {county_events["year"].max()}')

County-level records: 37,422
Unique counties: 2,824
Year range: 2020 - 2025


## Aggregate to county-month

In [5]:
agg_rules = {
    'EVENT_ID':           'count',
    'TOTAL_DAMAGE':       'sum',
    'DURATION_HOURS':     'sum',
}

monthly = (
    county_events
    .groupby(['stcofips', 'year', 'month'])
    .agg(agg_rules)
    .reset_index()
    .rename(columns={
        'EVENT_ID':       'event_count',
        'TOTAL_DAMAGE':   'total_damage',
        'DURATION_HOURS': 'total_duration_days',
    })
)

# Convert hours to days
monthly['total_duration_days'] = monthly['total_duration_days'] / 24.0

# Log damage — log(x + 1) handles zeros cleanly
monthly['log_damage'] = np.log1p(monthly['total_damage'])

print(f'County-month records: {len(monthly):,}')
print(f'Unique counties with storms: {monthly["stcofips"].nunique():,}')
monthly.head()

County-month records: 14,660
Unique counties with storms: 2,824


,stcofips,year,month,event_count,total_damage,total_duration_days,log_damage
0,01001,2021,4,2,0.0,0.001389,0.0
1,01001,2021,5,1,0.0,0.001389,0.0
2,01001,2025,4,1,0.0,0.003472,0.0
3,01001,2025,5,1,0.0,0.000694,0.0
4,01001,2025,7,1,0.0,0.003472,0.0


## Validate

In [6]:
assert monthly.duplicated(['stcofips', 'year', 'month']).sum() == 0, 'Duplicate county-month rows'
assert monthly.isnull().sum().sum() == 0, 'Unexpected nulls'
assert monthly['event_count'].gt(0).all(), 'Zero event count rows'
assert monthly['log_damage'].ge(0).all(), 'Negative log damage'
assert monthly['stcofips'].str.len().eq(5).all(), 'FIPS not all 5 digits'
assert set(monthly['year'].unique()) == set(NOAA_URLS.keys()), 'Missing years'

print('All assertions passed')
print(f'\nRecords per year:')
print(monthly.groupby('year').size().reset_index(name='county_month_records'))
print(f'\nDamage summary:')
print(monthly[['total_damage', 'log_damage', 'event_count', 'total_duration_days']].describe().round(2))

All assertions passed

Records per year:
   year  county_month_records
0  2020                  2914
1  2021                  2433
2  2022                  2231
3  2023                  2713
4  2024                  2012
5  2025                  2357

Damage summary:
       total_damage  log_damage  event_count  total_duration_days
count  1.466000e+04    14660.00     14660.00             14660.00
mean   2.757595e+05        0.87         2.55                 0.01
std    1.249858e+07        2.95         3.18                 0.18
min    0.000000e+00        0.00         1.00                 0.00
25%    0.000000e+00        0.00         1.00                 0.00
50%    0.000000e+00        0.00         1.00                 0.00
75%    0.000000e+00        0.00         3.00                 0.00
max    1.300200e+09       20.99        69.00                20.01


## Export

In [7]:
output_cols = [
    'stcofips', 'year', 'month',
    'event_count', 'total_damage', 'log_damage', 'total_duration_days'
]

monthly = monthly[output_cols].sort_values(['stcofips', 'year', 'month']).reset_index(drop=True)

monthly.to_pickle('../data/processed/storm_events.pkl')

print(f'Exported storm_events.pkl')
print(f'Shape: {monthly.shape}')
print(f'Columns: {monthly.columns.tolist()}')

Exported storm_events.pkl
Shape: (14660, 7)
Columns: ['stcofips', 'year', 'month', 'event_count', 'total_damage', 'log_damage', 'total_duration_days']
